# FF3 Security Selection — 2000-01 ~ 2010-12 (t(α) 순)

산출물은 `outputs/ff3_selected.csv` 하나입니다. 작업 폴더 아래 `data/` 에 `IFE_term_stock_data.csv` 와 `F-F_Research_Data_Factors.csv`(또는 압축 그대로의 `_CSV.zip`)를 두세요. Colab 에서는 첫 코드 셀의 Drive 마운트 두 줄 주석을 풀고 `BASE` 를 바꾸면 됩니다.

In [ ]:
# =========================================================================
# Fama-French 3-Factor 기반 Security Selection — 단일 구간 2000-01 ~ 2010-12
# =========================================================================
# 산출물: outputs/ff3_selected.csv 
#
# 설계
#   구간      : 2000-01 ~ 2010-12 (주가 132개월, 월수익률 131개)
#   유니버스  : 구간 내내 상장을 유지한 종목 (balanced panel)
#               - 구간의 모든 월에 ADJ_PRC 관측치가 있고 모든 월수익률이 유효
#               - 2010-12 시점 EXCHCD ∈ {1,2,3} (NYSE/AMEX/NASDAQ)
#   회귀      : R_i - RF = α_i + b_i(Mkt-RF) + s_i·SMB + h_i·HML + e_i  (구간 전체 1회)
#   게이트    : α > 0 AND t(α) ≥ 2 AND 유동성 하위 20% 제외 AND sigma guard 통과
#   정렬      : t(α) 내림차순, 상한 없음 (게이트 통과 전체가 결과물)
#
# 2011년 이후 정보 차단
#
# 입력 (data/ 폴더)
#   IFE_term_stock_data.csv
#   F-F_Research_Data_Factors.csv  ( 또는 압축을 풀지 않은 _CSV.zip)
#
#  ADJ_PRC 는 배당 재투자가 반영된 조정가격(총수익률 기준)
# =========================================================================

import os                                             # 경로 확인·폴더 생성
import zipfile                                        # 팩터 파일이 zip 으로만 있을 때 직접 열기
import numpy as np                                    # 행렬 연산 (OLS)
import pandas as pd                                   # 데이터프레임 처리
from scipy import stats                               # t-분포로 p-value 계산

In [ ]:
# -------------------------------------------------------------------------
# 0. 경로 및 파라미터
# -------------------------------------------------------------------------
BASE = (os.path.dirname(os.path.abspath(__file__))    # .py 로 실행하면 스크립트가 있는 폴더
        if "__file__" in globals() else os.getcwd())  # 노트북이면 현재 작업 폴더
DATA_DIR = os.path.join(BASE, "data")                 # 입력 파일 폴더
OUTPUT_DIR = os.path.join(BASE, "outputs")            # 결과 폴더
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "ff3_selected.csv")  # 유일한 산출물 경로

SAMPLE_START = pd.Period("2000-01", freq="M")         # 분석 시작월
SAMPLE_END = pd.Period("2010-12", freq="M")           # 분석 종료월 (이후 데이터는 사용 안 함)

RANK_METRIC = "t_alpha"                               # 정렬 기준: t(α)
T_CUT = 2.0                                           # 게이트: t(α) 하한
LIQ_CUT = 0.20                                        # 게이트: 구간 평균 거래량 하위 20% 제외
VALID_EXCHCD = [1, 2, 3]                              # 허용 거래소: 1=NYSE, 2=AMEX, 3=NASDAQ
RET_CAP = 5.0                                         # 월수익률 +500% 초과는 데이터 오류로 보고 제거

DEDUP_BY_COMPANY = True                               # 같은 회사 복수 클래스(예: HEICO)는 거래량 큰 쪽만 남김
TRACKING_STOCK_NAMES = {"APPLERA CORP", "LOEWS CORP"} # 트래킹 주식: 회사명이 같아도 다른 사업부라 합치지 않음

SIGMA_FLOOR_ABS = 0.015                               # sigma guard: 월 표준편차 1.5% 미만이면 비현실적
ZERO_RET_MAX = 0.30                                   # sigma guard: 수익률 0 인 달이 30% 초과면 가격 미갱신 의심
SHARPE_CAP = 3.0                                      # sigma guard: |Sharpe| 3 초과이면서
SIGMA_TAIL_PCTILE = 0.05                              #              표준편차가 하위 5% 이면 분모 폭주로 판단

In [ ]:
# -------------------------------------------------------------------------
# 보조 함수
# -------------------------------------------------------------------------
def find_input(candidates):                           # 후보 파일명 중 실제 존재하는 첫 경로를 찾는다
    for folder in (DATA_DIR, BASE):                   # data/ 폴더를 먼저, 없으면 BASE 바로 아래를 본다
        for name in candidates:                       # 후보 파일명을 순서대로 확인
            path = os.path.join(folder, name)         # 전체 경로 조립
            if os.path.exists(path):                  # 파일이 있으면
                return path                           # 그 경로를 돌려준다
    raise FileNotFoundError(                          # 끝까지 못 찾으면 어디를 봤는지 알려주고 중단
        f"{candidates} 를 {DATA_DIR} 또는 {BASE} 에서 찾지 못함")  # 오류 메시지


def read_ff_text(path):                               # 팩터 파일 원문을 문자열로 읽는다
    if path.lower().endswith(".zip"):                 # zip 파일이면
        with zipfile.ZipFile(path) as zf:             # 압축을 열고
            inner = [n for n in zf.namelist()         # 안에 든 파일 중
                     if n.lower().endswith(".csv")][0]  # 첫 번째 CSV 를 골라
            return zf.read(inner).decode("latin-1")   # 바이트를 문자열로 바꿔 반환
    with open(path, encoding="latin-1") as f:         # 일반 CSV 면
        return f.read()                               # 그대로 읽어 반환


def load_ff3(path):                                   # French 원본 포맷 → 월별 팩터 DataFrame
    rows = []                                         # 월별 행을 모을 리스트
    for ln in read_ff_text(path).splitlines():        # 파일을 한 줄씩 순회
        p = [x.strip() for x in ln.split(",")]        # 쉼표로 나누고 공백 제거
        if len(p) == 5 and p[0].isdigit() and len(p[0]) == 6:  # YYYYMM 으로 시작하는 월별 행만 (연간 블록은 4자리라 제외)
            rows.append(p)                            # 채택
    ff = pd.DataFrame(rows, columns=["ym", "MKT_RF", "SMB", "HML", "RF"])  # 열 이름 부여
    ff["ym"] = pd.PeriodIndex(ff["ym"].str[:4] + "-" + ff["ym"].str[4:], freq="M")  # 'YYYYMM' → 월 Period
    return ff.set_index("ym").astype(float) / 100.0   # 월을 인덱스로, % 단위를 소수로

In [ ]:
# -------------------------------------------------------------------------
# 1. 주가 데이터 로드 — 2010-12 이후는 즉시 삭제
# -------------------------------------------------------------------------
PATH_STOCK = find_input(["IFE_term_stock_data.csv"])  # 주가 파일 위치 확인
raw = pd.read_csv(PATH_STOCK, parse_dates=["date"])   # 주가 패널 읽기 (date 를 날짜형으로)
raw["ym"] = raw["date"].dt.to_period("M")             # 날짜 → 월 Period
raw = raw[(raw["ym"] >= SAMPLE_START) & (raw["ym"] <= SAMPLE_END)].copy()  # 2000-01~2010-12 만 남김
assert raw["ym"].max() <= SAMPLE_END, "2011년 이후 주가가 남아 있음"  # 2011년 이후 행이 없는지 확인
raw = raw.drop_duplicates(subset=["PERMNO", "ym"], keep="first")  # 같은 종목·같은 월 중복 제거
raw = raw[raw["ADJ_PRC"] > 0]                         # 가격 결측·0 이하 행 제거
raw = raw.sort_values(["PERMNO", "ym"]).reset_index(drop=True)  # 종목·월 순으로 정렬
raw["mnum"] = raw["ym"].dt.year * 12 + raw["ym"].dt.month  # 월을 정수로 바꿈 (연속월 판정용)
print(f"[1] 주가 {len(raw):,}행 ({SAMPLE_START}~{SAMPLE_END})")  # 진행 상황 출력

In [ ]:
# -------------------------------------------------------------------------
# 2. 팩터 로드 — 마찬가지로 2010-12 까지만
# -------------------------------------------------------------------------
PATH_FF = find_input(["F-F_Research_Data_Factors.csv",  # 압축 푼 파일 (소문자 확장자)
                      "F-F_Research_Data_Factors.CSV",  # 압축 푼 파일 (대문자 확장자)
                      "F-F_Research_Data_Factors_CSV.zip"])  # 압축 그대로
ff3 = load_ff3(PATH_FF).loc[SAMPLE_START:SAMPLE_END]  # 팩터를 읽고 분석 구간만 남김
assert ff3.index.max() <= SAMPLE_END, "2011년 이후 팩터가 남아 있음"  # 2011년 이후 행이 없는지 확인
print(f"[2] 팩터 {ff3.index.min()}~{ff3.index.max()} ({len(ff3)}개월)")  # 진행 상황 출력

In [ ]:
# -------------------------------------------------------------------------
# 3. Balanced panel: 구간 내내 상장된 종목
# -------------------------------------------------------------------------
n_months = len(pd.period_range(SAMPLE_START, SAMPLE_END, freq="M"))  # 구간 월 수 = 132
n_rets = n_months - 1                                 # 첫 달은 기준가라 수익률은 131개

price_cnt = raw.groupby("PERMNO")["ym"].nunique()     # 종목별 주가가 있는 월 수
full_price = price_cnt[price_cnt == n_months].index   # 132개월 모두 주가가 있는 종목
sub = raw[raw["PERMNO"].isin(full_price)].copy()      # 그 종목만 남김

grp = sub.groupby("PERMNO")                           # 종목별로 묶음
gap = sub["mnum"] - grp["mnum"].shift(1)              # 직전 관측과의 월 간격
prev_prc = grp["ADJ_PRC"].shift(1)                    # 직전 월 가격
sub["RET"] = np.where(gap == 1,                       # 직전 관측이 정확히 1개월 전일 때만
                      sub["ADJ_PRC"] / prev_prc - 1.0,  # 월수익률 = 이번 달 가격 / 지난달 가격 - 1
                      np.nan)                         # 아니면 결측
sub.loc[sub["RET"] > RET_CAP, "RET"] = np.nan         # +500% 초과 수익률은 데이터 오류로 보고 결측

ret_cnt = sub.dropna(subset=["RET"]).groupby("PERMNO").size()  # 종목별 유효 수익률 개수
balanced = ret_cnt[ret_cnt == n_rets].index           # 131개 모두 유효한 종목

at_end = sub[sub["ym"] == SAMPLE_END].set_index("PERMNO")  # 2010-12 행 (식별정보·거래소 판단 기준)
exch_ok = at_end.index[at_end["EXCHCD"].isin(VALID_EXCHCD)]  # 2010-12 에 NYSE/AMEX/NASDAQ 상장
balanced = balanced.intersection(exch_ok)             # 두 조건을 모두 만족하는 종목
panel = sub[sub["PERMNO"].isin(balanced)].dropna(subset=["RET"]).copy()  # 분석용 수익률 패널

ident = at_end.loc[balanced, ["TICKER", "COMNAM", "EXCHCD"]]  # 종목명·티커·거래소는 2010-12 값
avg_vol = panel.groupby("PERMNO")["VOL"].mean()       # 구간 평균 거래량 (유동성·중복제거 기준)

if DEDUP_BY_COMPANY:                                  # 같은 회사의 복수 클래스를 하나로 정리
    tmp = ident[["COMNAM"]].assign(avg_vol=avg_vol.reindex(ident.index))  # 회사명 + 평균 거래량
    is_track = tmp["COMNAM"].isin(TRACKING_STOCK_NAMES)  # 트래킹 주식 여부
    dups = tmp[~is_track & tmp["COMNAM"].duplicated(keep=False)]  # 트래킹이 아닌데 회사명이 겹치는 종목
    dups = dups.sort_values(["COMNAM", "avg_vol"], ascending=[True, False])  # 회사별 거래량 큰 순
    keep = dups.drop_duplicates("COMNAM", keep="first").index  # 회사별 거래량 1위 클래스만 유지
    drop = dups.index.difference(keep)                # 나머지 클래스는 제외 대상
    panel = panel[~panel["PERMNO"].isin(drop)]        # 패널에서 제외
    ident = ident.drop(drop)                          # 식별정보에서도 제외
print(f"[3] 최종 유니버스 {panel['PERMNO'].nunique():,}종목")  # 진행 상황 출력

In [ ]:
# -------------------------------------------------------------------------
# 4. 종목별 FF3 회귀 (모든 종목이 같은 X 를 쓰므로 행렬로 한 번에 계산)
# -------------------------------------------------------------------------
R = panel.pivot(index="ym", columns="PERMNO", values="RET")  # 월 × 종목 수익률 행렬
months = R.index                                      # 회귀에 쓰는 월 (2000-02~2010-12)
assert R.notna().all().all() and len(months) == n_rets  # 결측 없는 131개월인지 확인
assert months.max() == SAMPLE_END                     # 회귀 마지막 월이 2010-12 인지 확인
F = ff3.loc[months, ["MKT_RF", "SMB", "HML"]].to_numpy()  # 팩터 행렬 (T×3)
rf = ff3.loc[months, "RF"].to_numpy()                 # 무위험수익률 (T)
Y = R.to_numpy() - rf[:, None]                        # 초과수익률 R - RF (T×N)

T = len(months)                                       # 관측 월 수
X = np.column_stack([np.ones(T), F])                  # 절편 열을 붙인 설명변수 (T×4)
XtX_inv = np.linalg.inv(X.T @ X)                      # (X'X)^-1
B = XtX_inv @ X.T @ Y                                 # OLS 계수 (4×N): α, b, s, h
E = Y - X @ B                                         # 잔차
s2 = (E ** 2).sum(axis=0) / (T - 4)                   # 잔차분산 (자유도 = T - 계수 4개)
se = np.sqrt(np.outer(np.diag(XtX_inv), s2))          # 계수별 표준오차 (4×N)
Tst = B / se                                          # t-통계량
r2 = 1 - (E ** 2).sum(axis=0) / ((Y - Y.mean(axis=0)) ** 2).sum(axis=0)  # 결정계수 R²

res = pd.DataFrame({                                  # 종목별 결과 표
    "PERMNO": R.columns,                              # 종목 고유번호
    "alpha": B[0], "b_mkt": B[1],                     # 알파(월), 시장 베타
    "s_smb": B[2], "h_hml": B[3],                     # SMB 노출, HML 노출
    "t_alpha": Tst[0], "t_mkt": Tst[1],               # t(α), t(b)
    "t_smb": Tst[2], "t_hml": Tst[3],                 # t(s), t(h)
    "r2": r2,                                         # 결정계수
    "resid_sd": np.sqrt(s2),                          # 잔차 표준편차 (월)
    "se_alpha": se[0],                                # 알파 표준오차 (월)
    "nobs": T,                                        # 관측치 수
    "ex_mean_m": Y.mean(axis=0),                      # 월평균 초과수익률
    "ex_sd_m": Y.std(axis=0, ddof=1),                 # 월 초과수익률 표본표준편차
    "zero_ret_share": (np.abs(R.to_numpy()) < 1e-9).mean(axis=0),  # 수익률이 0 인 달의 비중
})                                                    # 표 생성 끝
res = res.join(ident, on="PERMNO")                    # 티커·회사명·거래소 붙이기 (2010-12 값)
res["avg_vol"] = res["PERMNO"].map(avg_vol)           # 구간 평균 거래량 붙이기
res["alpha_ann"] = res["alpha"] * 12                  # 연율화 알파 (단순 ×12)
res["p_alpha_one"] = stats.t.sf(res["t_alpha"], df=T - 4)  # α>0 단측 p-value (다중검정 미보정)
res["appraisal_ratio"] = res["alpha"] / res["resid_sd"]   # α / 잔차표준편차
res["sharpe_raw"] = res["ex_mean_m"] / res["ex_sd_m"] * np.sqrt(12)  # 연율 Sharpe (guard 적용 전)

In [ ]:
# -------------------------------------------------------------------------
# 5. Sigma guard · 유동성 · 게이트
# -------------------------------------------------------------------------
sd_tail = res["ex_sd_m"].quantile(SIGMA_TAIL_PCTILE)  # 표준편차 하위 5% 경계값
flag = ((res["ex_sd_m"] < SIGMA_FLOOR_ABS)            # 표준편차가 비현실적으로 작거나
        | (res["zero_ret_share"] > ZERO_RET_MAX)      # 가격 미갱신이 의심되거나
        | ~np.isfinite(res["sharpe_raw"])             # Sharpe 가 계산 불가이거나
        | ((res["sharpe_raw"].abs() > SHARPE_CAP)     # Sharpe 가 지나치게 크면서
           & (res["ex_sd_m"] < sd_tail)))             # 그 원인이 작은 분모인 경우
res["sharpe_valid"] = ~flag                           # guard 통과 여부
res["sharpe"] = res["sharpe_raw"].where(res["sharpe_valid"])  # 걸린 종목의 Sharpe 는 결측 처리

res["liq_pass"] = res["avg_vol"] >= res["avg_vol"].quantile(LIQ_CUT)  # 거래량 하위 20% 가 아니면 통과
res["gate_pass"] = ((res["alpha"] > 0)                # 알파가 양수이고
                    & (res["t_alpha"] >= T_CUT)       # t(α) 가 2 이상이고
                    & res["liq_pass"]                 # 유동성 필터를 통과하고
                    & res["sharpe"].notna())          # sigma guard 도 통과

In [ ]:
# -------------------------------------------------------------------------
# 6. t(α) 순 정렬 및 선정
# -------------------------------------------------------------------------
res = res.sort_values(RANK_METRIC, ascending=False).reset_index(drop=True)  # 전체를 t(α) 내림차순 정렬
res.insert(0, "rank_all", np.arange(1, len(res) + 1)) # 전체 유니버스 기준 순위
sel = res[res["gate_pass"]].copy()                    # 게이트 통과 종목만
sel.insert(0, "rank", np.arange(1, len(sel) + 1))     # 통과 종목 내 순위
print(f"[6] 게이트 통과 {len(sel)}종목 / 유니버스 {len(res):,}종목")  # 진행 상황 출력

In [ ]:
# -------------------------------------------------------------------------
# 7. 저장 — ff3_selected.csv 하나만
# -------------------------------------------------------------------------
os.makedirs(OUTPUT_DIR, exist_ok=True)                # 결과 폴더가 없으면 만든다
sel.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")  # 저장 (엑셀에서 열어도 깨지지 않도록 BOM 포함)
saved = pd.read_csv(OUTPUT_FILE)                      # 저장된 파일을 다시 읽어서
assert len(saved) == len(sel), "저장된 행 수가 다름"  # 행 수가 맞는지 확인
assert saved["t_alpha"].is_monotonic_decreasing, "t(α) 정렬이 깨짐"  # 정렬이 유지됐는지 확인
print(f"✔️ 저장 완료: {OUTPUT_FILE} ({len(saved)}행)")  # 최종 경로와 행 수 출력
print(sel[["rank", "TICKER", "COMNAM", "t_alpha", "alpha_ann"]].head(10).round(3).to_string(index=False))  # 상위 10개 미리보기